[DeepSeek tool call][1]

<!-- Reference -->
[1]: https://api-docs.deepseek.com/zh-cn/guides/tool_calls/

In [1]:
# 1. Load library dependencies
from openai import OpenAI
from dotenv import load_dotenv
import os

In [2]:
# 2. load environment variable from .env file


load_dotenv()

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

In [3]:
client = OpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
)


def send_messages(messages):
    response = client.chat.completions.create(
        model="deepseek-flash",
        messages=messages,
        tools=tools,
        extra_body={"thinking": {"type": "disabled"}},
    )
    return response.choices[0].message


tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get weather of a location, the user should supply a location first.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    },
                },
                "required": ["location"],
            },
        },
    },
]

messages = [{"role": "user", "content": "How's the weather in Hangzhou, Zhejiang?"}]


In [4]:
message = send_messages(messages)
print(f"User>\t {messages[0]['content']}")


User>	 How's the weather in Hangzhou, Zhejiang?


In [10]:
message.model_dump()


{'content': "I'll check the weather in Hangzhou for you.",
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': [{'id': 'call_00_hJIdtjXQGBhGi87BOpfH5938',
   'function': {'arguments': '{"location": "Hangzhou, Zhejiang"}',
    'name': 'get_weather'},
   'type': 'function',
   'index': 0}]}

In [16]:
message.tool_calls[0].model_dump()

{'id': 'call_00_hJIdtjXQGBhGi87BOpfH5938',
 'function': {'arguments': '{"location": "Hangzhou, Zhejiang"}',
  'name': 'get_weather'},
 'type': 'function',
 'index': 0}

In [17]:
messages.append(message)


In [20]:
messages

[{'role': 'user', 'content': "How's the weather in Hangzhou, Zhejiang?"},
 ChatCompletionMessage(content="I'll check the weather in Hangzhou for you.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_00_hJIdtjXQGBhGi87BOpfH5938', function=Function(arguments='{"location": "Hangzhou, Zhejiang"}', name='get_weather'), type='function', index=0)])]

In [21]:
messages.append({"role": "tool", "tool_call_id": tool.id, "content": "24℃"})


In [22]:
message = send_messages(messages)


In [25]:
message.model_dump()

{'content': "The current temperature in Hangzhou, Zhejiang is **24°C** (about 75°F) — pleasantly mild.\n\nUnfortunately, the weather service only returned the temperature, so I don't have details on conditions (sunny, cloudy, rain) or humidity. Let me know if you'd like me to check anything else!",
 'refusal': None,
 'role': 'assistant',
 'annotations': None,
 'audio': None,
 'function_call': None,
 'tool_calls': None}

In [26]:
print(f"Model>\t {message.content}")



Model>	 The current temperature in Hangzhou, Zhejiang is **24°C** (about 75°F) — pleasantly mild.

Unfortunately, the weather service only returned the temperature, so I don't have details on conditions (sunny, cloudy, rain) or humidity. Let me know if you'd like me to check anything else!
